# 📡 TP Télécom 3GPP — Phase 5 : RAFT (RAG + Fine-Tuning)

**Objectif** : Combiner RAG et Fine-Tuning pour obtenir le meilleur des deux mondes.

**Principe RAFT :**
```
Documents 3GPP → RAG → Contexte enrichi
                              ↓
              Fine-Tuning sur (Question + Contexte RAG + Réponse)
                              ↓
                  Modèle spécialisé 3GPP + RAG-aware
```

**Comparaison finale :**
```
LLM Seul → RAG → Fine-Tuning → RAFT
```

```
Phase 1 ✅ → Phase 2 ✅ → Phase 3 ✅ → Phase 4 ✅ → [Phase 5] → Phase 6 → Phase 7
```

## 1. Installation des dépendances

In [ ]:
!pip install -q transformers torch peft trl accelerate
!pip install -q faiss-cpu sentence-transformers rank-bm25
!pip install -q evaluate rouge_score datasets
!pip install -q pandas matplotlib
print('✅ Installation terminée')

## 2. Imports & Configuration

In [ ]:
import json, time, os, pickle
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig
from sentence_transformers import SentenceTransformer
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR = r'C:\Users\HP\Documents\TP-LLM-3GPP-Pipeline'
print('✅ Imports effectués')

## 3. Chargement des Ressources (Phases précédentes)

In [ ]:
# Config Phase 1
with open(f'{SAVE_DIR}\\pipeline_config.json') as f:
    config = json.load(f)
MODEL_ID = config['best_model_id']
print(f'✅ Modèle : {config["best_model_name"]}')

# Dataset
TRAIN_PATH = r'C:\Users\HP\Downloads\TeleQnA_training.txt'
TEST_PATH  = r'C:\Users\HP\Downloads\TeleQnA_testing.txt'
with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    train_data = json.load(f)
with open(TEST_PATH, 'r', encoding='utf-8') as f:
    test_data = json.load(f)
print(f'✅ Dataset : {len(train_data)} train | {len(test_data)} test')

# Modèle d'embedding
print('🔄 Chargement embedding model...')
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('✅ Embedding model chargé')

## 4. Construction du Corpus et Index FAISS

In [ ]:
# Construction corpus
corpus = []
for key, item in train_data.items():
    answer_idx  = item.get('answer', 1)
    answer_text = item.get(f'option {answer_idx}', str(answer_idx))
    corpus.append({
        'text'    : f"Question: {item['question']} Answer: {answer_text}. {item.get('explanation', '')}",
        'category': item.get('category', '3GPP')
    })

# Index FAISS
texts      = [doc['text'] for doc in corpus]
embeddings = embed_model.encode(texts, batch_size=64, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')
faiss.normalize_L2(embeddings)
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f'✅ Corpus : {len(corpus)} documents')
print(f'✅ Index FAISS : {index.ntotal} vecteurs')

def retrieve_context(query, top_k=3):
    q_emb = embed_model.encode([query]).astype('float32')
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, top_k)
    return [corpus[i]['text'] for i in indices[0]]

## 5. Création du Dataset RAFT

In [ ]:
# RAFT = Fine-Tuning sur des exemples (Question + Contexte RAG + Réponse)
print('🔄 Création du dataset RAFT...')
raft_texts = []
items = list(train_data.values())[:200]

for item in items:
    answer_idx  = item.get('answer', 1)
    answer_text = item.get(f'option {answer_idx}', str(answer_idx))
    question    = item['question']

    # Récupération du contexte RAG
    contexts = retrieve_context(question, top_k=2)
    context  = ' | '.join([c[:150] for c in contexts])

    # Format RAFT : Question + Contexte + Réponse
    raft_text = (
        f"Context: {context} "
        f"Question: {question} "
        f"Answer: {answer_text}. "
        f"{item.get('explanation', '')}"
    )
    raft_texts.append({'text': raft_text})

raft_dataset = Dataset.from_list(raft_texts)
print(f'✅ Dataset RAFT créé : {len(raft_dataset)} exemples')
print(f'\nExemple RAFT :')
print(f'  {raft_dataset[0]["text"][:300]}...')

## 6. Fine-Tuning RAFT

In [ ]:
# Chargement modèle
print('🔄 Chargement du modèle...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
print('✅ Modèle chargé')

# Configuration LoRA
lora_config = LoraConfig(
    task_type    = TaskType.CAUSAL_LM,
    r            = 4,
    lora_alpha   = 8,
    lora_dropout = 0.1,
    bias         = 'none',
    target_modules = ['c_attn']
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA : {100*trainable/total:.2f}% paramètres entraînables')

# Configuration entraînement RAFT
sft_config = SFTConfig(
    output_dir                  = f'{SAVE_DIR}\\Phase5\\checkpoints',
    num_train_epochs            = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate               = 2e-4,
    fp16                        = False,
    logging_steps               = 10,
    save_steps                  = 50,
    save_total_limit            = 1,
    report_to                   = 'none',
    use_cpu                     = True,
    dataset_text_field          = 'text',
    max_length                  = 256
)

trainer = SFTTrainer(
    model            = model,
    args             = sft_config,
    train_dataset    = raft_dataset,
    processing_class = tokenizer
)

print('🔄 Début du Fine-Tuning RAFT...')
print('   ⚠️  Sur CPU cela prend environ 10-20 minutes')
start = time.time()
trainer.train()
elapsed = time.time() - start
print(f'\n✅ RAFT Fine-Tuning terminé en {elapsed/60:.1f} minutes')

model.save_pretrained(f'{SAVE_DIR}\\Phase5\\model_raft')
tokenizer.save_pretrained(f'{SAVE_DIR}\\Phase5\\model_raft')
print('💾 Modèle RAFT sauvegardé → Phase5/model_raft')

## 7. Comparaison Finale : LLM → RAG → Fine-Tuning → RAFT

In [ ]:
# Chargement des 4 approches
print('🔄 Chargement des modèles pour comparaison...')

# 1. LLM Base
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
base_pipe  = pipeline('text-generation', model=base_model,
                       tokenizer=tokenizer, pad_token_id=50256)

# 2. Modèle Fine-Tuné (Phase 4)
ft_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
ft_model = PeftModel.from_pretrained(ft_model, f'{SAVE_DIR}\\Phase4\\model_finetuned')
ft_pipe  = pipeline('text-generation', model=ft_model,
                     tokenizer=tokenizer, pad_token_id=50256)

# 3. Modèle RAFT
raft_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
raft_model = PeftModel.from_pretrained(raft_model, f'{SAVE_DIR}\\Phase5\\model_raft')
raft_pipe  = pipeline('text-generation', model=raft_model,
                       tokenizer=tokenizer, pad_token_id=50256)

print('✅ Tous les modèles chargés')

# Questions de test
test_items = list(test_data.values())[:10]
results    = []

for i, item in enumerate(test_items):
    question    = item['question']
    answer_idx  = item.get('answer', item.get('option 1', ''))
    answer_text = item.get(f'option {answer_idx}', str(answer_idx)) if isinstance(answer_idx, int) else str(answer_idx)

    # Contexte RAG
    contexts = retrieve_context(question, top_k=2)
    context  = ' | '.join([c[:150] for c in contexts])

    # 1. LLM seul
    p1     = f'3GPP Question: {question} Answer:'
    r1     = base_pipe(p1, max_new_tokens=80, do_sample=False, pad_token_id=50256, truncation=True)
    ans_llm = r1[0]['generated_text'].replace(p1, '').strip()

    # 2. RAG
    p2     = f'Context: {context} Question: {question} Answer:'
    r2     = base_pipe(p2, max_new_tokens=80, do_sample=False, pad_token_id=50256, truncation=True)
    ans_rag = r2[0]['generated_text'].replace(p2, '').strip()

    # 3. Fine-Tuning
    r3     = ft_pipe(p1, max_new_tokens=80, do_sample=False, pad_token_id=50256, truncation=True)
    ans_ft  = r3[0]['generated_text'].replace(p1, '').strip()

    # 4. RAFT
    r4      = raft_pipe(p2, max_new_tokens=80, do_sample=False, pad_token_id=50256, truncation=True)
    ans_raft = r4[0]['generated_text'].replace(p2, '').strip()

    results.append({
        'id'        : i + 1,
        'question'  : question,
        'reference' : str(answer_text),
        'llm_only'  : ans_llm,
        'rag'       : ans_rag,
        'finetuned' : ans_ft,
        'raft'      : ans_raft
    })
    print(f'  ✓ Q{i+1} traité')

df_results = pd.DataFrame(results)
df_results.to_csv(f'{SAVE_DIR}\\Phase5\\phase5_results.csv', index=False)
print(f'\n✅ {len(df_results)} résultats sauvegardés')

## 8. Évaluation ROUGE — Comparaison des 4 Approches

In [ ]:
from evaluate import load as load_metric
rouge = load_metric('rouge')
refs  = df_results['reference'].tolist()

approches = [
    ('LLM Seul',      'llm_only'),
    ('RAG',           'rag'),
    ('Fine-Tuning',   'finetuned'),
    ('RAFT',          'raft')
]

eval_results = []
for name, col in approches:
    scores = rouge.compute(predictions=df_results[col].tolist(), references=refs)
    eval_results.append({
        'Approche': name,
        'ROUGE-1' : round(scores['rouge1'], 4),
        'ROUGE-2' : round(scores['rouge2'], 4),
        'ROUGE-L' : round(scores['rougeL'], 4)
    })
    print(f'✅ {name} évalué')

df_eval = pd.DataFrame(eval_results)
print('\n📊 Comparaison finale des 4 approches :')
print(df_eval.to_string(index=False))
df_eval.to_csv(f'{SAVE_DIR}\\Phase5\\phase5_evaluation.csv', index=False)
print('\n💾 Sauvegardé → Phase5/phase5_evaluation.csv')

## 9. Visualisation Finale

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
palette = ['#FF7043', '#26A69A', '#1565C0', '#7B1FA2']
metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
x, w    = np.arange(len(metrics)), 0.2

for i, (_, row) in enumerate(df_eval.iterrows()):
    bars = ax.bar(x + i*w, [row[m] for m in metrics],
                  width=w, label=row['Approche'], color=palette[i], alpha=0.85)
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.001,
                f'{b.get_height():.4f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xticks(x + w*1.5)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_title('Comparaison Finale : LLM → RAG → Fine-Tuning → RAFT\nDataset 3GPP',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Score ROUGE')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}\\Phase5\\phase5_comparaison_finale.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Graphique → Phase5/phase5_comparaison_finale.png')

---
## ✅ Phase 5 Terminée — RAFT

**Ce qu'on a accompli :**
- ✅ Dataset RAFT créé (Question + Contexte RAG + Réponse)
- ✅ Fine-Tuning sur exemples enrichis par RAG
- ✅ Comparaison des 4 approches : LLM → RAG → FT → RAFT

**Fichiers produits :**
- `Phase5/model_raft/` — Modèle RAFT
- `Phase5/phase5_results.csv`
- `Phase5/phase5_evaluation.csv`
- `Phase5/phase5_comparaison_finale.png`

**➡️ Prochaine étape : Phase 6 — RAG + Agent**